# `numpy`中的一元多项式(`poly1d`)对象, 以及支持的运算

In [1]:
#以别名形式导入numpy. 
import numpy as np; 
#导入与ndarray有关的数据类型. 
from numpy import int8, int16, int32, int64; 
from numpy import uint8, uint16, uint32, uint64; 
from numpy import float16, float32, float64; 
from numpy import complex64, complex128; 

In [2]:
from numpy import polynomial as nppn

## `numpy.poly1d`对象的构造
* 注意: `np.poly1d`只能用于定义标量的多项式, **不能**用于定义**矩阵多项式**. 
* 如将矩阵作为变量代入多项式, 会[计算**每个元素**在多项式下的值](#计算多项式的值)

### 使用系数列表构造多项式
用法: 
```python
px = np.poly1d(iter_coef)
```
* `iter_coef` 包含多项式系数的迭代器
* 支持的迭代器类型: 
    * `list`, `tuple`, `numpy.ndarray`
    * 迭代器中不能内嵌迭代器; `numpy.ndarray`的`ndim`属性必须为1
* 迭代器内的所有元素应当为数值型对象, 否则所构建的多项式, 在被调用部分方法时将报错
    * Python内建的, 或`numpy`模块中定义的各类`int`, `float`, `complex`; 
    * `Decimal`, `Fraction`
    * 当迭代器为`list`或`tuple`时, 其中的元素类型不需要完全一致
* 系数按照自变量**次数从高到低**排列
* 当多项式构造期间使用的迭代器长度为`n`时, 下标为`i`的元素 \
    **表示`(n - i - 1)`次项**的系数
* 除最高次项以外的其他任意项不存在时, 对应的系数为0, 必须列出
* 构造完成后, 首非零元(不含)之前的元素将被移除
* 构造过程支持`variable = varstr`选项, 用于指定使用`print`语句将所构造的`nunpy.poly1d`对象\
    打印到控制台/输出单元时, 自变量的名称. 

In [3]:
#定义一系列物理量
grav_Accl = 9.80665; #地表重力加速度(单位: m/s¹)

In [4]:
#计算自由落体运动的速度(单位: m/s)与时间(单位: s)的关系
velo_free_fall = np.poly1d([grav_Accl, 0], variable="t"); 
print(velo_free_fall); #注意到打印的结果预留了一行, 这一行用于显示二次(含)以上的项的幂指数
#如果需要通过同一条print语句顺次打印多个poly1d对象, 需要指定sep="\n"

 
9.807 t


### 使用零点列表构造多项式
用法: 
```python
px = np.poly1d(iter_root, r=True)
```
* `iter_root` 包含多项式零点的迭代器
* 迭代器结构和元素类型要求与"[使用系数列表构造多项式](#使用系数列表构造多项式)"所用的迭代器相同
* 构造的多项式为$$\prod \limits_{i = 0}^{n - 1} (x - \mathrm{iter\_root}[i]),~\mathrm{其中} n = len(\mathrm{iter\_root})$$
    * 所构造多项式的最高次项系数为1
    * 多项式次数为构造期间使用的迭代器长度
* 迭代器中可以包含重复的元素, 同一元素重复$k$次, 表示所构造的多项式中对应的零点为$k$重零点
* 元素的顺序对所构造多项式的结果无影响

In [5]:
#构造零点在[-2, 2]内随机分布的多项式
if "random" not in globals(): import random; 
if "oper" not in globals(): import operator as oper; 
if "reduce" not in globals(): from functools import reduce; 
#随机生成16个零点
poly_root = [random.uniform(-2, 2) for i in range(16)]; 
print(poly_root); 
#利用零点构造16阶多项式
poly_High_Deg = np.poly1d(poly_root, r=True); 
print(poly_High_Deg); 
#利用零点构造多项式的每个因式
poly_Factor = [np.poly1d([1, -a]) for a in poly_root]; 
#将所有因式连乘(不能使用np.prod)
poly_Expand = reduce(oper.mul, poly_Factor); 
print(poly_High_Deg == poly_Expand)

[-0.9606067522226738, 1.9922859050912476, -0.43027094784770936, 1.5550229144602454, 0.8827588461109972, 0.6052328968184835, -1.212088798827243, 0.6204767986666422, 0.39532743447424723, 0.21580106304150704, 1.8625021251108667, -0.9799062348333609, 0.47922331910355487, 0.4496929191172949, 0.7751791192525195, -0.0676499741037353]
   16         15         14         13         12         11        10
1 x  - 6.183 x  + 11.03 x  + 5.134 x  - 37.02 x  + 29.43 x  + 24.5 x 
         9         8         7         6         5         4          3
 - 47.5 x + 13.54 x + 17.41 x - 14.71 x + 2.525 x + 1.507 x - 0.7926 x
          2
 + 0.123 x - 0.001041 x - 0.0009053
True


## `numpy.poly1d`对象的调用和修改

### 计算多项式的值
* 使用`np.polyval`方法求多项式`poly`在自变量为`x`时的值: 
    ```python
    np.polyval(poly, x)
    ```
    * 求值过程通过秦九韶-Horner算法实现. 
    * 支持向量化运算. 如果`x`是`list`, `tuple`, `range`, `np.ndarray`等类型, `np.polyval`将对`x`中的每个元素求多项式的值, 并返回同型的`np.ndarray`对象. 
* `numpy.poly1d`对象可作为一元函数使用, 当`np.poly1d`被挂载至`poly`时, 
    ```python
    poly(x)
    ```
    与` np.polyval(poly, x)`对传入参数的要求, 以及返回的结果均相同, 但`poly`可直接作为一元函数使用, 在部分场合下具有更高的可读性. 

In [6]:
#计算自由落体的物体自释放后1s, 2s, 3s末的速度
print(np.polyval(velo_free_fall, range(1, 4))); #使用np.polyval方法
print(velo_free_fall(range(1, 4))); #将np.poly1d直接视为一元函数

[ 9.80665 19.6133  29.41995]
[ 9.80665 19.6133  29.41995]
